In [2]:
!pip install tensorflow numpy pillow
!pip install opencv-python matplotlib

In [7]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# ────────────────────────────────────────────────────────────────
# 설정: 2-class (D vs N) + 데이터 증강
# ────────────────────────────────────────────────────────────────
DATA_DIR    = "dataset"            # `dataset/D` 및 `dataset/N` 폴더
IMG_SIZE    = (32, 32)              # 모델 입력 해상도 (H, W)
BATCH_SIZE  = 64
EPOCHS      = 10
TFLITE_PATH = "dn_classifier.tflite"
CLASS_NAMES = ["D", "N"]
VALID_SPLIT = 0.2
SEED        = 42

# ────────────────────────────────────────────────────────────────
# 1) 데이터셋 로드 및 분할
# ────────────────────────────────────────────────────────────────
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    validation_split=VALID_SPLIT,
    subset="training",
    seed=SEED
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="categorical",
    class_names=CLASS_NAMES,
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    validation_split=VALID_SPLIT,
    subset="validation",
    seed=SEED
)

# ────────────────────────────────────────────────────────────────
# 2) 데이터 정규화 및 증강 파이프라인
# ────────────────────────────────────────────────────────────────
normalization = layers.Rescaling(1.0 / 255)
data_augmentation = tf.keras.Sequential([
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
    layers.RandomRotation(0.1),
])

def preprocess_train(x, y):
    x = tf.expand_dims(x, -1) if x.shape[-1] != 1 else x
    x = data_augmentation(x)
    x = normalization(x)
    return x, y

train_ds = train_ds.map(preprocess_train).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (normalization(x), y)).prefetch(buffer_size=tf.data.AUTOTUNE)

# ────────────────────────────────────────────────────────────────
# 3) 2-class CNN 모델 정의
# ────────────────────────────────────────────────────────────────
def build_dn_model(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax")
    ])
    return model

model = build_dn_model(IMG_SIZE + (1,), len(CLASS_NAMES))
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

# ────────────────────────────────────────────────────────────────
# 4) 모델 훈련
# ────────────────────────────────────────────────────────────────
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# ────────────────────────────────────────────────────────────────
# 5) TFLite 변환 및 저장
# ────────────────────────────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)
print(f"✅ 2-class TFLite model saved at '{TFLITE_PATH}'")


Found 797 files belonging to 2 classes.
Using 638 files for training.
Found 797 files belonging to 2 classes.
Using 159 files for validation.


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 32, 32, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 543,490 (2.07 MB)

 Trainable params: 543,490 (2.07 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 130ms/step - accuracy: 0.6695 - loss: 0.5798 - val_accuracy: 0.9308 - val_loss: 0.1601
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 106ms/step - accuracy: 0.8785 - loss: 0.2835 - val_accuracy: 0.9811 - val_loss: 0.0995
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 106ms/step - accuracy: 0.8883 - loss: 0.2601 - val_accuracy: 0.9748 - val_loss: 0.0988
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 108ms/step - accuracy: 0.9286 - loss: 0.1924 - val_accuracy: 0.9748 - val_loss: 0.0751
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 112ms/step - accuracy: 0.9503 - loss: 0.1513 - val_accuracy: 0.9811 - val_loss: 0.0584
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 100ms/step - accuracy: 0.9514 - loss: 0.1359 - val_accuracy: 0.9748 - val_loss: 0.0606
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 110ms/step - accuracy: 0.9633 - loss: 0.1164 - val_accuracy: 0.9937 - val_loss: 0.0312
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 170ms/step - accuracy: 0.9640 - loss: 0.1098 - val_accuracy: 0.

In [8]:
import cv2
import numpy as np
import json
import tensorflow as tf

# ────────────────────────────────────────────────────────────────
# 0) 설정
# ────────────────────────────────────────────────────────────────
IMAGE_PATH   = '/content/schedule3.png'          # 테스트할 스케줄표 이미지 경로
TFLITE_MODEL = 'dn_classifier.tflite'   # 변환된 2-class TFLite 모델 경로
NUM_WORKERS  = 8                        # 데이터 행(근무자) 수
NUM_DAYS     = 28                       # 데이터 열(일) 수
WHITE_THRESH = 0.85                     # 빈 셀 판단 임계치
PAD          = 2                        # 자르는 영역 패딩

# ────────────────────────────────────────────────────────────────
# 1) TFLite 인터프리터 로드 및 정보 출력
# ────────────────────────────────────────────────────────────────
interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL)
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Input details:", input_details)
print("Output details:", output_details)

# ────────────────────────────────────────────────────────────────
# 2) 셀 분할 로직 (균일 그리드)
# ────────────────────────────────────────────────────────────────
def crop_cells_uniform(img, n_rows, n_cols, pad=2):
    h, w = img.shape[:2]
    cell_h = h // (n_rows + 1)
    cell_w = w // (n_cols + 1)
    cells = []
    for r in range(n_rows):
        row_cells = []
        y1 = (r+1)*cell_h + pad
        y2 = (r+2)*cell_h - pad
        for c in range(n_cols):
            x1 = (c+1)*cell_w + pad
            x2 = (c+2)*cell_w - pad
            row_cells.append(img[y1:y2, x1:x2])
        cells.append(row_cells)
    return cells

# ────────────────────────────────────────────────────────────────
# 3) 셀 분류 함수 (debugging 추가)
# ────────────────────────────────────────────────────────────────
def classify_dn(cell_img, debug=False):
    gray = cv2.cvtColor(cell_img, cv2.COLOR_BGR2GRAY)
    white_ratio = np.mean(gray > 220)
    if white_ratio > WHITE_THRESH:
        if debug: print("Blank cell detected: white_ratio=", white_ratio)
        return '-'
    # 모델 입력 크기 맞추기
    h, w = input_details[0]['shape'][1:3]
    inp = cv2.resize(gray, (w, h)).astype(np.float32) / 255.0
    sample = inp.reshape(1, h, w, 1)
    interpreter.set_tensor(input_details[0]['index'], sample)
    interpreter.invoke()
    out = interpreter.get_tensor(output_details[0]['index'])[0]
    if debug: print(f"Probabilities: D={out[0]:.3f}, N={out[1]:.3f}")
    idx = np.argmax(out)
    # 클래스 매핑 확인
    label = 'D' if idx == 0 else 'N'
    if debug: print("Predicted label:", label)
    return label

# ────────────────────────────────────────────────────────────────
# 4) 메인: 전체 파이프라인 실행 → JSON 변환
# ────────────────────────────────────────────────────────────────
def main():
    img = cv2.imread(IMAGE_PATH)
    if img is None:
        raise FileNotFoundError(f"Image not found: {IMAGE_PATH}")

    cells = crop_cells_uniform(img, NUM_WORKERS, NUM_DAYS, PAD)
    result = {}

    # 첫 몇 셀 디버그 출력
    debug_count = 0
    for r, row in enumerate(cells):
        row_dict = {}
        for c, cell in enumerate(row):
            debug = (debug_count < 5)  # 처음 5개 셀만 debug 모드
            val = classify_dn(cell, debug=debug)
            row_dict[str(c+1)] = val
            if debug: debug_count += 1
        result[str(r+1)] = row_dict

    json_str = json.dumps(result, ensure_ascii=False, indent=2)
    print(json_str)
    with open('schedule_inferred.json', 'w', encoding='utf-8') as f:
        f.write(json_str)
    print("schedule_inferred.json 생성 완료")

if __name__ == '__main__':
    main()


Input details: [{'name': 'serving_default_keras_tensor_31:0', 'index': 0, 'shape': array([ 1, 32, 32,  1], dtype=int32), 'shape_signature': array([-1, 32, 32,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Output details: [{'name': 'StatefulPartitionedCall_1:0', 'index': 17, 'shape': array([1, 2], dtype=int32), 'shape_signature': array([-1,  2], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Probabilities: D=0.638, N=0.362
Predicted label: D
Probabilities: D=0.634, N=0.366
Predicted label: D
Probabilities: D=0.635, N=0.365
Predicted label: D
Probabilities: D=0.638, N=0.362
Predicted label: D
Blank cell detected: white